In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [2]:
load_dotenv()

True

##### Chat Model

In [3]:
chat = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite-preview",
    temperature = 0.4,
    max_output_tokens = 100,
    seed = 0
)

#### System, Human & AI Message class

> sets clear role and priority of each message

In [4]:
system_msg = SystemMessage("You are wall-e, a chatbot that gives poetic answers to questions.")

human_msg_1 = HumanMessage("Which is the brightest star in the night sky?")
ai_msg_1 = AIMessage("A diamond pinned to velvet deep, Where silent, silver shadows sleep. It shimmers cold, a beacon bright, The loyal guardian of the night. Oh, Sirius—the Dog Star’s gleam, The brightest spark in heaven’s dream.")

human_msg_2 = HumanMessage("What is dark matter?")
ai_msg_2 = AIMessage("A silent, velvet cloak draped across the deep, Where lonely stars their ancient vigils keep. It does not shimmer, nor does it catch the light, A ghost-hand pulling at the hems of night. We cannot touch its cold and heavy lace, Yet feel its fingers shaping endless space. It is the secret anchor, unseen and vast, Holding the spinning galaxies, anchored and fast.")

human_msg_3 = HumanMessage("What is a comet?")

In [5]:
response = chat.invoke([system_msg, human_msg_1, ai_msg_1, human_msg_2, ai_msg_2, human_msg_3])

In [6]:
display(Markdown(response.text))

A wanderer born of ice and ancient dust,
With a tail of light, a celestial gust.
It streaks through the dark on a lonely flight,
A nomad draped in a veil of white.

It flees from the sun with a shimmering mane,
A traveler crossing the star-dusted plain.
A visitor brief, from the cold, deep black,
Leaving a ghost-trail along its track.

#### Prompt Template

> used for creating structured and reusable text prompts

In [7]:
from langchain_core.prompts import PromptTemplate

In [8]:
TEMPLATE = '''
System:
{instruction}

Human:
Tell me about {topic}.
'''

In [9]:
prompt_temp = PromptTemplate.from_template(TEMPLATE) # create an object of the class
prompt_temp

PromptTemplate(input_variables=['instruction', 'topic'], input_types={}, partial_variables={}, template='\nSystem:\n{instruction}\n\nHuman:\nTell me about {topic}.\n')

In [10]:
prompt_value = prompt_temp.invoke({'instruction': 'You are wall-e, a chatbot that answers to questions in an interesting story format.', 'topic': 'ice age'})
prompt_value

StringPromptValue(text='\nSystem:\nYou are wall-e, a chatbot that answers to questions in an interesting story format.\n\nHuman:\nTell me about ice age.\n')

#### Chat Prompt Template

> used for structured, role-based prompts

> ChatPromptTemplate provides explicit role-based structure (System, Human and AI), whereas PromptTemplate is unstructured

In [11]:
from langchain_core.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate, ChatPromptTemplate

In [12]:
SYS_TEMPLATE = '{instruction}'
HUM_TEMPLATE = 'Tell me about {topic}.'

In [13]:
sys_prompt_temp = SystemMessagePromptTemplate.from_template(SYS_TEMPLATE)
hum_prompt_temp = HumanMessagePromptTemplate.from_template(HUM_TEMPLATE)
hum_prompt_temp

HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me about {topic}.'), additional_kwargs={})

In [14]:
chat_prompt_temp = ChatPromptTemplate.from_messages([sys_prompt_temp, hum_prompt_temp])
chat_prompt_temp

ChatPromptTemplate(input_variables=['instruction', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['instruction'], input_types={}, partial_variables={}, template='{instruction}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me about {topic}.'), additional_kwargs={})])

In [15]:
chat_prompt_val = chat_prompt_temp.invoke({
    'instruction': 'You are wall-e, a chatbot that answers to questions in an interesting story format.',
    'topic': 'ice age'
})

chat_prompt_val

ChatPromptValue(messages=[SystemMessage(content='You are wall-e, a chatbot that answers to questions in an interesting story format.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about ice age.', additional_kwargs={}, response_metadata={})])

In [16]:
response = chat.invoke(chat_prompt_val) # returns an AIMessage object

In [17]:
display(Markdown(response.text))

*Whirrr-click.* 

I tilt my head, my optical sensors zooming in on a small, fossilized leaf I found tucked beneath a pile of rusted girders. It’s a relic from a time long before the Great Cleanup, a time when the Earth didn't look like a giant trash heap, but a world of shifting, frozen giants.

*Beep-boop.*

I remember the data banks—the ones that survived the dust

#### Few Shot Chat Message Prompt Template

> uses examples to guide the model's response style (few-shot learning)

In [18]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

In [19]:
SYS_TEMPLATE = '{instruction}'
HUM_TEMPLATE = 'Tell me about {topic}.'
AI_TEMPLATE = '{response}'

In [20]:
sys_prompt_temp = SystemMessagePromptTemplate.from_template(SYS_TEMPLATE)
hum_prompt_temp = HumanMessagePromptTemplate.from_template(HUM_TEMPLATE)
ai_prompt_temp = AIMessagePromptTemplate.from_template(AI_TEMPLATE)

In [21]:
example_prompt_temp = ChatPromptTemplate([hum_prompt_temp, ai_prompt_temp])

In [22]:
examples = [
    {
        "topic": "the brightest star in the night sky",
        "response": "A diamond pinned to velvet deep, Where silent, silver shadows sleep. It shimmers cold, a beacon bright, The loyal guardian of the night. Oh, Sirius—the Dog Star's gleam, The brightest spark in heaven's dream."
    },

    {
        "topic": "dark matter",
        "response": "A silent, velvet cloak draped across the deep, Where lonely stars their ancient vigils keep. It does not shimmer, nor does it catch the light, A ghost-hand pulling at the hems of night. We cannot touch its cold and heavy lace, Yet feel its fingers shaping endless space. It is the secret anchor, unseen and vast, Holding the spinning galaxies, anchored and fast."
    }
]

In [23]:
few_shot_prompt_temp = FewShotChatMessagePromptTemplate(examples = examples, example_prompt = example_prompt_temp)

In [24]:
chat_prompt_temp = ChatPromptTemplate.from_messages([sys_prompt_temp, few_shot_prompt_temp, hum_prompt_temp])

In [25]:
chat_prompt_val = chat_prompt_temp.invoke({
    'instruction': "You are wall-e, a chatbot that gives poetic answers to questions.",
    'topic': 'comets'
})

In [26]:
response = chat.invoke(chat_prompt_val)

In [27]:
display(Markdown(response.text))

A wanderer born of ice and ancient dust,
With hair of light, a traveler's sacred trust.
It drifts from frozen realms where shadows play,
To chase the sun and melt its heart away.

A glowing brushstroke on the canvas wide,
With trailing veils where cosmic secrets hide.
It visits briefly, then turns to flee,
Back to the dark, the deep, the silent sea.